# Attaching Data Assets (raw and processed)
- ophys:
    - Stage_0 (1 session): Natural movies
    - Stage_1 (3 sessions): drifting gratings

- coregistration:
    -   ophys_zstack:
        - Ophys functional (4 sessions, 8 planes) to ophys structure (1 3D image 0-400um) mapping
    - xenium_zstack:
        - Xenium to ophys strucure mapping
- xenium:
    - processed:
        - images
        - segmentation masks
        - cell locations
        - transcript locations
        - cellxgene table
    - cell_types:
        - cell types for each cell (based on ABC atlas)
        - cell typting measures

In [12]:
import sys
sys.path.append('/root/capsule/code/Preprocessing')

import codeocean
import codeocean.data_asset

from codeocean.data_asset import (
        DataAssetParams
        )
from codeocean.data_asset import DataAssetAttachParams
import aind_session
from aind_session.utils.codeocean_utils import get_data_asset_source_dir, get_codeocean_client, search_data_assets, get_data_asset_search_query, get_subject_data_assets
from analysis_utils import get_running_timestamps, get_running_df, get_stimulus_timestamps
from codeocean.data_asset import DataAssetAttachParams
import boto3
import json
import numpy as np
from pathlib import Path
import pandas as pd

import pickle
DATA_DIR = Path("/root/capsule/data")

In [13]:
client = get_codeocean_client()


In [14]:
xenium_mouse_ids = [778174,786297,797371]

co_assets_dict = {}
for mouse_id in xenium_mouse_ids:
    co_assets_dict[mouse_id] = {}
    co_assets_dict[mouse_id]['ophys'] = {}
    assets = list(get_subject_data_assets(mouse_id))
    assets.sort(key=lambda x: x.name)
    # search_params["query"] = get_data_asset_search_query(mouse_id=mouse_id)
    # from_co = search_data_assets(search_params) + search_data_assets(
    #     {"query": str(mouse_id)}
    # )
    assets_raw = [asset for asset in assets if 'multiplane-ophys' in asset.tags and 'raw' in asset.tags and 'multisession' not in asset.tags]

    for asset_raw in assets_raw:
        s3_dir = get_data_asset_source_dir(asset_raw.id).as_posix()
        s3 = boto3.resource('s3')
        BUCKET_NAME = s3_dir.split('/')[2]
        PREFIX = '/'.join(s3_dir.split('/')[3:]) + '/'
        KEY = PREFIX + 'session.json'
        obj = s3.Object(BUCKET_NAME, KEY)
        data = obj.get()['Body'].read()
        json_string = data.decode('utf-8')
        session_json = json.loads(json_string)
        session_type = session_json['session_type']
        if session_type == 'STAGE_0' or session_type == 'STAGE_1':
            asset_processed = [asset for asset in assets if f"{asset_raw.name}_processed" in asset.name][-1]
            co_assets_dict[mouse_id]['ophys'][asset_raw.name] = {'session_type':session_type,'raw':asset_raw,"processed":asset_processed} 

    ophys_asset_names =  list(co_assets_dict[mouse_id]['ophys'].keys())
    co_assets_dict[mouse_id]['ophys'] = {ophys_asset_name:co_assets_dict[mouse_id]['ophys'][ophys_asset_name] for ophys_asset_name in ophys_asset_names[:4]}
    for asset_name in co_assets_dict[mouse_id]['ophys']:
        co_assets_dict[mouse_id]['ophys'][asset_name]['glm'] = [asset for asset in assets if f"{asset_name}_glm" in asset.name][-1]
    
    co_assets_dict[mouse_id]['cortical_zstack'] = {}
    co_assets_dict[mouse_id]['cortical_zstack']['registered'] = [asset for asset in assets if asset.name.startswith(f'ophys-z-stacks_{mouse_id}') and asset.name.endswith('registered')][-1]
    co_assets_dict[mouse_id]['cortical_zstack']['segmented'] = [asset for asset in assets if asset.name.startswith(f'ophys-z-stacks_{mouse_id}') and asset.name.endswith('segmented_cpsam')][-1]
    
    co_assets_dict[mouse_id]['xenium'] = {}
    co_assets_dict[mouse_id]['xenium']['processed'] = [asset for asset in assets if asset.name.startswith("Xenium_") and asset.name.endswith("_processed")][-1]
    co_assets_dict[mouse_id]['xenium']['cell_types'] = [asset for asset in assets if asset.name.startswith("Xenium_") and asset.name.endswith("_mapped")][-1]

    co_assets_dict[mouse_id]['coregistration'] = {}
    co_assets_dict[mouse_id]['coregistration']['ophys_zstack'] = [asset for asset in assets if 'cortical-zstack-to-fov' in asset.tags][-1]
    co_assets_dict[mouse_id]['coregistration']['xenium_zstack'] = [asset for asset in assets if asset.name.startswith("Xenium_ophys") and 'coregistration' in asset.tags][-1]
    

co_assets = list(np.hstack([[a['raw'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a['processed'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a['glm'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a for a in co_assets_dict[mouse_id]['xenium'].values()]+\
[a for a in co_assets_dict[mouse_id]['cortical_zstack'].values()]+\
[a for a in co_assets_dict[mouse_id]['coregistration'].values()] for mouse_id in xenium_mouse_ids]))


# for asset in co_assets:
#     print(asset.id)


IndexError: list index out of range

In [20]:
assets = list(get_subject_data_assets(mouse_id))
assets.sort(key=lambda x: x.name)
[asset.name for asset in assets]

['731327_subject_nwb_hd5',
 'Example T1 and T2 MRI Images',
 'ROICaT_797371_U01',
 'SmartSPIM Alignment Subjects V7',
 'Subject-aligned CCF Masks V1',
 'Xenium_797371_2025-10-22',
 'Xenium_797371_2025-10-22_mapped',
 'Xenium_797371_2025-10-22_processed',
 'Xenium_797371_2025-10-22_processed',
 'Xenium_797371_2025-10-22_processed_2025-11-10_23-07-44',
 'Xenium_797371_2025-10-22_slides',
 'Xenium_ophys-z-zstack_co-registrated_797371',
 'Xenium_ophys_797371_coregistered',
 'confocal-thick-tissue_797371_2025-09-25',
 'confocal-thick-tissue_797371_2025-09-25_processed_2026-01-06_23-00-58',
 'confocal-thick-tissue_797371_2025-09-25_processed_2026-01-06_23-00-58',
 'multiplane-ophys_755252_2024-11-12_09-43-51_processed_2025-10-09_19-53-36',
 'multiplane-ophys_755252_2024-11-12_09-43-51_processed_2025-10-24_03-44-57',
 'multiplane-ophys_755252_2024-11-13_09-16-29_processed_2025-10-09_21-37-25',
 'multiplane-ophys_755252_2024-11-14_11-28-00_processed_2025-10-09_23-17-12',
 'multiplane-ophys_755

In [7]:
[asset for asset in assets if asset.name.startswith(f'ophys-z-stacks_{mouse_id}') and asset.name.endswith('registered')]

[]

In [ ]:
co_assets = list(np.hstack([[a for a in co_assets_dict[mouse_id]['xenium'].values()] for mouse_id in xenium_mouse_ids]))


for asset in co_assets:
    print(asset.id)

NameError: name 'xenium_mouse_ids' is not defined

In [4]:
xenium_data_info = {}

for mouse_id in xenium_mouse_ids:
    xenium_data_info[mouse_id] = {}
    xenium_data_info[mouse_id]['ophys'] = {}
    xenium_data_info[mouse_id]['xenium'] = {}
    xenium_data_info[mouse_id]['coregistration'] = {}
    xenium_data_info[mouse_id]['cortical_zstack'] = {}


    ophys_asset_names =  list(co_assets_dict[mouse_id]['ophys'].keys())
    xenium_data_info[mouse_id]['ophys'] = {f"session_{session_ind}":co_assets_dict[mouse_id]['ophys'][ophys_asset_name].copy() for session_ind, ophys_asset_name in enumerate(ophys_asset_names)}
    for session in xenium_data_info[mouse_id]['ophys'].keys():
        xenium_data_info[mouse_id]['ophys'][session]['raw'] = xenium_data_info[mouse_id]['ophys'][session]['raw'].name
        xenium_data_info[mouse_id]['ophys'][session]['processed'] = xenium_data_info[mouse_id]['ophys'][session]['processed'].name
        xenium_data_info[mouse_id]['ophys'][session]['glm'] = xenium_data_info[mouse_id]['ophys'][session]['glm'].name

    xenium_data_info[mouse_id]['xenium']['processed'] = co_assets_dict[mouse_id]['xenium']['processed'].name
    xenium_data_info[mouse_id]['xenium']['cell_types'] = co_assets_dict[mouse_id]['xenium']['cell_types'].name
    xenium_data_info[mouse_id]['coregistration']['ophys_zstack'] = co_assets_dict[mouse_id]['coregistration']['ophys_zstack'].name
    xenium_data_info[mouse_id]['coregistration']['xenium_zstack'] = co_assets_dict[mouse_id]['coregistration']['xenium_zstack'].name
    xenium_data_info[mouse_id]['cortical_zstack']['registered'] = co_assets_dict[mouse_id]['cortical_zstack']['registered'].name
    xenium_data_info[mouse_id]['cortical_zstack']['segmented'] = co_assets_dict[mouse_id]['cortical_zstack']['segmented'].name

xenium_data_info
with open('/root/capsule/code/Preprocessing/xenium_data_info.pkl', 'wb') as f:
    pickle.dump(xenium_data_info, f)
